In [1]:
from ftplib import FTP, error_perm, error_temp
import os
from urllib.error import URLError
import urllib.request
import boto3
from botocore.exceptions import ClientError

In [ ]:
# Mock de variáveis de ambiente do .env
regiao = 'us-east-1'

# Cria o client para o Amazon S3
try:
    s3_client = boto3.client('s3', region_name=regiao)

except ClientError as e:
    print(f'Não foi possível criar o client: {e}')

In [ ]:
# --------------------
# FUNÇÃO
# --------------------
def mapear_arquivos_ftp() -> dict:
    '''
    Cria um dicionário contendo a competência, no formato aamm, dos arquivos mais recentes disponibilizados no módulo 
    Profissionais do CNES, no servidor FTP do DataSUS, bem como uma lista com os nomes dos arquivos dessa competência.

    Returns:
        dict_arquivos(dict): Dicionário com a competência mais recente e a lista de arquivos. 
    '''
    # Dicionário que armazenará o resultado da função
    dict_arquivos = {}

    # Variáveis de host e diretório
    host = 'ftp.datasus.gov.br'
    dir_pf = 'dissemin/publicos/CNES/200508_/Dados/PF/'

    print(f'Iniciando conexão com o servidor {host}')

    try:
        with FTP(host) as servidor:
            
            # Acessa o servidor FTP do DataSUS como usuário anônimo
            servidor.login()
            print('Conexão estabelecida.')

            # Navega até o diretório que contém as bases de dados
            servidor.cwd(dir_pf)
            print(f'Navegando para o diretório {dir_pf}')
            
            # Armazena no dicionário a competência atual dos arquivos do servidor FTP (formato: aamm)
            dict_arquivos['competencia'] = int(servidor.nlst()[-1][-8:-4])
            print(f'Competência atual (FTP): {dict_arquivos['competencia']}')

            # Cria a string de filtragem dos arquivos
            filtro = '*' + str(dict_arquivos['competencia']) + '.dbc'
            print(f'String de filtro criada: {filtro}')

            # Armazena no dicionário a lista com os arquivos da competência mais atual do servidor FTP
            dict_arquivos['arquivos'] = servidor.nlst(filtro)

        return dict_arquivos

    except error_perm as e:
        # Logs de erros de caráter permanente (inexistência de diretórios e/ou arquivos)
        print(f'Falha permanente no servidor FTP: {e}')
        return None

    except error_temp as e:
        # Logs de erros de caráter temporário (instabilidade no servidor)
        print(f'Falha temporária no servidor FTP: {e}')        
        return None

    except Exception as e:
        # Logs de error gerais, armazenando a mensagem completa do erro.
        print(f'Erro: {e}')   
        return None


# --------------------
# EXECUÇÃO
# --------------------
resposta_funcao = mapear_arquivos_ftp()
print(f'Retorno da função: {resposta_funcao}')

Iniciando conexão com o servidor ftp.datasus.gov.br
Conexão estabelecida.
Navegando para o diretório dissemin/publicos/CNES/200508_/Dados/PF/
Competência atual (FTP): 2603
String de filtro criada: *2603.dbc
Retorno da função: {'competencia': 2603, 'arquivos': ['PFAC2603.dbc', 'PFAL2603.dbc', 'PFAM2603.dbc', 'PFAP2603.dbc', 'PFBA2603.dbc', 'PFCE2603.dbc', 'PFDF2603.dbc', 'PFES2603.dbc', 'PFGO2603.dbc', 'PFMA2603.dbc', 'PFMG2603.dbc', 'PFMS2603.dbc', 'PFMT2603.dbc', 'PFPA2603.dbc', 'PFPB2603.dbc', 'PFPE2603.dbc', 'PFPI2603.dbc', 'PFPR2603.dbc', 'PFRJ2603.dbc', 'PFRN2603.dbc', 'PFRO2603.dbc', 'PFRR2603.dbc', 'PFRS2603.dbc', 'PFSC2603.dbc', 'PFSE2603.dbc', 'PFSP2603.dbc', 'PFTO2603.dbc']}


In [ ]:
# --------------------
# MOCK DE DADOS
# --------------------
bucket = 'pipeline-cnes-profissionais'


# --------------------
# FUNÇÃO
# --------------------
def verificar_bucket(bucket:str)->bool:
    '''
    Verifica se o bucket informado existe no Amazon S3.

    Args:
        bucket(str): Nome do bucket no Amazon S3.

    Returns:
        existe(bool): Retorna True se o bucket existir e False se não existir.
    '''
    print(f'Bucket de consulta: {bucket}')

    # Retorna os metadados dos buckets no Amazon S3
    resposta_s3 = s3_client.list_buckets()
    print(f'Resposta Amazon S3: {resposta_s3}')
    
    # Cria uma lista com os nomes dos buckets existentes
    lista_buckets = [bucket_s3['Name'] for bucket_s3 in resposta_s3['Buckets']]
    print(f'Buckets existentes: {lista_buckets}')
    
    if bucket in lista_buckets:
        return True
    else:
        return False
    
    
# --------------------
# EXECUÇÃO
# --------------------
resposta_funcao = verificar_bucket(bucket)
print(f'Retorno da Função (Bucket existe?): {resposta_funcao}')

Bucket de consulta: pipeline-cnes-profissionais
Resposta Amazon S3: {'ResponseMetadata': {'RequestId': 'MF070G2P8WN13KQ8', 'HostId': '0vGnp/YUO23hUTj96Uooht2UvkyIundWuSpzN20row6b3/f7gR3CbKkgJLoNIwTxLYtr24lNbuppFslDBMAzzpGs7+R5c7BK', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': '0vGnp/YUO23hUTj96Uooht2UvkyIundWuSpzN20row6b3/f7gR3CbKkgJLoNIwTxLYtr24lNbuppFslDBMAzzpGs7+R5c7BK', 'x-amz-request-id': 'MF070G2P8WN13KQ8', 'date': 'Sat, 02 May 2026 10:57:38 GMT', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'Buckets': [{'Name': 'pipeline-cnes-profissionais', 'CreationDate': datetime.datetime(2026, 4, 23, 16, 7, 33, tzinfo=tzutc()), 'BucketArn': 'arn:aws:s3:::pipeline-cnes-profissionais'}, {'Name': 'teste-bucket1sta', 'CreationDate': datetime.datetime(2026, 5, 1, 0, 54, 18, tzinfo=tzutc()), 'BucketArn': 'arn:aws:s3:::teste-bucket1sta'}], 'Owner': {'ID': '2087122cf78dbecbb2a0f01c15bde2118c52a1ac4ea4fd26aa6a9e7f26496832'}}
B

In [ ]:
# --------------------
# MOCK DE DADOS
# --------------------
bucket = 'pipeline-cnes-profissionais'
regiao = 'us-east-1'


# --------------------
# FUNÇÃO
# --------------------
def criar_bucket(bucket:str, regiao:str=regiao)->bool:
    '''
    Cria um bucket no Amazon S3.

    Args:
        bucket(str): Nome do bucket que será criado.
        regiao(str): Região em que o bucket será criado. Padrão: Região informada o .env.

    Returns:
        criacao(bool): Retorna True se o bucket foi criado e False se não foi criado.
    '''
    print(f'Nome do Bucket: {bucket}')
    print(f'Região: {regiao}')

    try:                    
        # Cria o bucket na região especificada no .env
        if regiao == 'us-east-1':
            resposta_s3 = s3_client.create_bucket(Bucket=bucket)
            print(f'Resposta Amazon: {resposta_s3}')

        else:
            resposta_s3 = s3_client.create_bucket(
                Bucket=bucket,
                CreateBucketConfiguration={
                    'LocationConstraint': regiao
                }
            )
            print(f'Resposta Amazon: {resposta_s3}')

        if resposta_s3['ResponseMetadata']['HTTPStatusCode'] == 200:
            criacao = True
            return criacao
        
        else:
            print(f'Não foi possível criar o bucket {bucket}: {resposta_s3}')
            criacao = False
            return criacao

    except ClientError as e:
            print(f'Erro ao criar o bucket {bucket}: {e}')
            criacao = False
            return criacao


# --------------------
# EXECUÇÃO
# --------------------
resposta_funcao = criar_bucket(bucket, regiao)
print(f'Retorno da função (Bucket criado?): {resposta_funcao}')

Nome do Bucket: pipeline-cnes-profissionais
Região: us-east-1
Resposta Amazon: {'ResponseMetadata': {'RequestId': 'YAP8VZPF6DCJ97CZ', 'HostId': 'iMEmA0Dz4dNFGAmnxxyuYUy0pkODLMPBOKfhlc2BLtqhYa17B1wwwiQU455lG9I41g7OYAk2I0Y3NxL5ZdMpUWgF84QeQMR6', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'iMEmA0Dz4dNFGAmnxxyuYUy0pkODLMPBOKfhlc2BLtqhYa17B1wwwiQU455lG9I41g7OYAk2I0Y3NxL5ZdMpUWgF84QeQMR6', 'x-amz-request-id': 'YAP8VZPF6DCJ97CZ', 'date': 'Sat, 02 May 2026 11:04:26 GMT', 'location': '/pipeline-cnes-profissionais', 'x-amz-bucket-arn': 'arn:aws:s3:::pipeline-cnes-profissionais', 'content-length': '0', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'Location': '/pipeline-cnes-profissionais', 'BucketArn': 'arn:aws:s3:::pipeline-cnes-profissionais'}
Retorno da função (Bucket criado?): True


In [ ]:
# --------------------
# MOCK DE DADOS
# --------------------
bucket = 'pipeline-cnes-profissionais'


# --------------------
# FUNÇÃO
# --------------------
def mapear_arquivos_bucket(bucket:str) -> dict:
    '''
    Cria um dicionário contendo a competência, no formato aamm, dos arquivos mais recentes disponibilizados no bucket 
    do Amazon S3, bem como uma lista com o nome dos arquivos dessa competência.

    Args:
        bucket(str): Nome do bucket de destino no Amazon S3.

    Returns:
        dict_arquivos(dict): Dicionário com a competência mais recente e a lista de arquivos. 
    '''
    
    print(f'Nome do Bucket: {bucket}')
    dict_arquivos = {}

    # Consulta os arquivos existentes na camada bronze do bucket
    resposta_s3 = s3_client.list_objects_v2(Bucket=bucket, Prefix='profissionais/bronze/')
    print(f'Resposta Amazon: {resposta_s3}')
    
    if 'Contents' in resposta_s3:
        
        # Cria uma lista contendo os nomes dos arquivos da camada bronze
        arquivos_bucket = [arquivo['Key'] for arquivo in resposta_s3['Contents']]

        # Identifica a competência dos arquivos na camada bronze
        competencia = arquivos_bucket[-1][-8:-4]

        # Insere a competência e a lista com o nome dos arquivos no dicionário
        dict_arquivos['competencia'] = int(competencia)
        dict_arquivos['arquivos'] = arquivos_bucket
    
    else:
        dict_arquivos['competencia'] = 0
        dict_arquivos['arquivos'] = []

    return dict_arquivos


# --------------------
# EXECUÇÃO
# --------------------
resposta_funcao = mapear_arquivos_bucket(bucket)
print(f'Retorno da função: {resposta_funcao}')

Nome do Bucket: pipeline-cnes-profissionais
Resposta Amazon: {'ResponseMetadata': {'RequestId': 'B0REXY5V12198WJV', 'HostId': 'B3SVFzzh9nPTfq2ga6+izlvfSaqD0qK9uZzGiyjFSvQQ2sxAyC0VGHvcVG2ZW0Yw1hg8Ly3CmcZ9g+tDHtDweTpZFSrYq1j5', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'B3SVFzzh9nPTfq2ga6+izlvfSaqD0qK9uZzGiyjFSvQQ2sxAyC0VGHvcVG2ZW0Yw1hg8Ly3CmcZ9g+tDHtDweTpZFSrYq1j5', 'x-amz-request-id': 'B0REXY5V12198WJV', 'date': 'Sat, 02 May 2026 11:29:03 GMT', 'x-amz-bucket-region': 'us-east-1', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'profissionais/bronze/PFAC2602.dbc', 'LastModified': datetime.datetime(2026, 5, 2, 11, 28, 29, tzinfo=tzutc()), 'ETag': '"7e0940963ad8444461a5e599692d908c"', 'ChecksumAlgorithm': ['CRC32'], 'ChecksumType': 'FULL_OBJECT', 'Size': 1234087, 'StorageClass': 'STANDARD'}, {'Key': 'profissionais/bronze/PFAL2602.dbc', 'LastModified': datetime.datetime(2026

In [21]:
# --------------------
# MOCK DE DADOS
# --------------------
arquivos = ['profissionais/bronze/PFAC2602.dbc', 'profissionais/bronze/PFAL2602.dbc']
bucket = 'pipeline-cnes-profissionais'


# --------------------
# FUNÇÃO
# --------------------
def excluir_arquivos_bucket(arquivos:list, bucket:str):
    '''
    Exclui os arquivos do bucket no Amazon S3.

    Args:
        arquivos(list): Lista de arquivos a serem excluídos do bucket. 
        bucket(str): Nome do bucket de destino no Amazon S3.
    '''

    print(f'Nome do bucket: {bucket}')
    print(f'Arquivos para deletar: {arquivos}')

    # Cria uma lista de dicionários dos arquivos a serem excluídos
    arquivos_deletar = [{'Key': arquivo} for arquivo in arquivos]

    try:
        # Exclui os arquivos do bucket
        s3_client.delete_objects(
            Bucket=bucket,
            Delete={
                'Objects': arquivos_deletar,
                'Quiet': True
            }
        )
        print(f'Arquivos excluídos com sucesso.')

    except ClientError as e:
        print(f'Erro ao tentar excluir os arquivos: {e}')


# --------------------
# EXECUÇÃO
# --------------------
excluir_arquivos_bucket(arquivos, bucket)

Nome do bucket: pipeline-cnes-profissionais
Arquivos para deletar: ['profissionais/bronze/PFAC2602.dbc', 'profissionais/bronze/PFAL2602.dbc']
Arquivos excluídos com sucesso.


In [ ]:
# --------------------
# MOCK DE DADOS
# --------------------
arquivos = ['PFAC2602.dbc', 'PFAL2602.dbc']
bucket = 'pipeline-cnes-profissionais'


# --------------------
# FUNÇÃO
# --------------------
def transferir_ftp_para_s3(arquivos:list, bucket:str):
    '''
    Transfere os arquivos do CNES Profissionais do servidor FTP do DataSUS para a camada bronze do
    bucket no Amazon S3.

    Args:
        arquivos(list): Lista contendo o nome dos arquivos que serão baixados.
        bucket(str): Nome do bucket de destino no Amazon S3.
    '''

    print(f'Nome do Bucket: {bucket}')
    print(f'Arquivos para baixar: {arquivos}')

    # URL base do servidor FTP
    url_ftp = 'ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/PF'

    # Contabiliza a quantidade de arquivos baixados
    cont_downloads = 0

    # Cria uma lista de arquivos que não tiveram a transferência concluída para o S3
    arquivos_erro = []

    # Realiza o download de cada arquivo da lista arquivos
    for arquivo in arquivos:

        try:
            # Cria uma requisição HTTPS para baixar o arquivo
            with urllib.request.urlopen(f'{url_ftp}/{arquivo}') as arquivo_path:

                print(f'Baixando arquivo: {arquivo}')

                # Nome do arquivo que será armazenado na camada bronze no bucket
                nome_objeto = f'profissionais/bronze/{arquivo}'

                # Faz o upload do arquivo no bucket do Amazon S3
                s3_client.upload_fileobj(
                    Fileobj=arquivo_path,
                    Bucket=bucket,
                    Key=nome_objeto
                )
                print(f'Download concluído: {arquivo}')
                
            # Incrementa a quantidade de downloas realizados
            cont_downloads += 1
                
        except URLError as e:
            # Inclui o nome do arquivo na lista de arquivos que deram erro
            arquivos_erro.append(arquivo)
            print(f'Não foi possível baixar o arquivo {arquivo}: {e}')
            continue

    print(f'Transferência concluída. Arquivos baixados: {cont_downloads}/{len(arquivos)}.')

    if cont_downloads != len(arquivos):
        print(f'Arquivos não baixados: {arquivos_erro}.')


# --------------------
# EXECUÇÃO
# --------------------
transferir_ftp_para_s3(arquivos, bucket)

Nome do Bucket: pipeline-cnes-profissionais
Arquivos para baixar: ['PFAC2602.dbc', 'PFAL2602.dbc']
Baixando arquivo: PFAC2602.dbc
Download concluído: PFAC2602.dbc
Baixando arquivo: PFAL2602.dbc
Download concluído: PFAL2602.dbc
Transferência concluída. Arquivos baixados: 2/2.
